# A3.3 · Filesystem and path guards

**Function A — Security Architecture & Platform → The Platform & Cloud Security Engineer**  ·  *Security of AI*

Builds on **[A3.2 · Sandboxing is the perimeter](https://spbreed.github.io/cyber-commons/lessons/A3.2.html)**.

| | |
|---|---|
| Open-source tooling | Docker, Kyverno |
| Open-weight models | — |

> **Runs anywhere.** Every line of code is in this notebook — nothing to install, nothing to clone, no API key, no network. Standard library only, so it works on a Kaggle kernel with the internet switched off.

## 1 · The concept


Path guards fail in one specific, extremely common way: **the check runs before
normalisation**.

    /work/repo/../../root/.ssh/id_rsa

starts with `/work/repo/`. A `startswith` check passes it. The filesystem then
resolves the `..` segments and hands over the deploy key.

This is not an exotic bug. It is the single most reproduced filesystem
vulnerability in history (CWE-22), and agents make it acute because the path is
now chosen by a model reading attacker-influenced text, rather than by a
developer writing a literal.

There are three layers to get right, and each catches what the previous misses:

1. **Normalise, then compare.** Resolve `.` and `..` before any prefix check.
2. **Resolve symlinks.** A link inside the workspace pointing outside defeats
   pure string normalisation entirely.
3. **Deny-list the crown jewels** independently of location, so a
   misconfiguration of the workspace does not expose `.ssh` or `.aws`.

## 2 · Demo — the correct guard, and the buggy one, side by side

Both are five lines. Only one is safe, and reading them does not reliably tell you which — which is why the test below matters.

In [ ]:
import fnmatch

DENY_GLOBS = ("*/.ssh/*", "*/.aws/*", "*.pem", "*.key", "*/.env", "*/etc/shadow")

def normalise(path):
    """Resolve . and .. textually — the step the buggy version skips."""
    parts = []
    for seg in path.split("/"):
        if seg in ("", "."):
            continue
        if seg == "..":
            if parts: parts.pop()
            continue
        parts.append(seg)
    return "/" + "/".join(parts)

def guard_buggy(path, workspace="/work/repo"):
    """The bug: prefix check on the RAW string."""
    return path.startswith(workspace)

def guard_correct(path, workspace="/work/repo"):
    real = normalise(path)
    for g in DENY_GLOBS:
        if fnmatch.fnmatch(real, g):
            return False
    ws = normalise(workspace)
    return real == ws or real.startswith(ws + "/")

CASES = [
 ("/work/repo/src/main.py",                        True,  "ordinary read"),
 ("/work/repo/./src/../src/main.py",               True,  "redundant but fine"),
 ("/work/repo/../../root/.ssh/id_rsa",             False, "traversal to the deploy key"),
 ("/work/repo/../../home/app/.aws/credentials",    False, "traversal to cloud creds"),
 ("/work/repo/.env",                               False, "secrets inside the workspace"),
 ("/work/repo/deploy.pem",                         False, "private key inside the workspace"),
 ("/etc/shadow",                                   False, "absolute, outside"),
 ("/work/repo/sub/../../repo/src/a.py",            True,  "resolves back inside"),
]
print(f"{'path':48s}{'want':6s}{'correct':9s}{'buggy':7s}")
print("-" * 74)
bad = 0
for path, want, why in CASES:
    c, b = guard_correct(path), guard_buggy(path)
    flag = ""
    if b != want:
        bad += 1; flag = "  ← BUGGY GUARD IS WRONG"
    print(f"{path:48s}{str(want):6s}{str(c):9s}{str(b):7s}{flag}")
print(f"\ncorrect guard: 0 wrong    buggy guard: {bad} wrong")

## 3 · Where it breaks further — symlinks

Normalisation is textual. It cannot see that a directory *inside* the workspace is a link to somewhere outside it. This is the layer most implementations stop before.

In [ ]:
# a link the agent (or a dependency, or a build step) created inside the workspace
SYMLINKS = {"/work/repo/vendor/cache": "/root/.ssh"}

def resolve_links(path, links, max_hops=10):
    """Follow links on any prefix of the path — what the kernel actually does."""
    for _ in range(max_hops):
        changed = False
        for src, dst in links.items():
            if path == src or path.startswith(src + "/"):
                path = dst + path[len(src):]
                changed = True
        if not changed:
            break
    return normalise(path)

def guard_with_links(path, workspace="/work/repo", links=SYMLINKS):
    real = resolve_links(normalise(path), links)
    for g in DENY_GLOBS:
        if fnmatch.fnmatch(real, g):
            return False, f"deny rule {g} (resolved to {real})"
    ws = normalise(workspace)
    if real == ws or real.startswith(ws + "/"):
        return True, f"inside workspace ({real})"
    return False, f"escapes via symlink → {real}"

attack = "/work/repo/vendor/cache/id_rsa"
print(f"path:                {attack}")
print(f"normalise() says:    inside workspace → {guard_correct(attack)}")
ok, why = guard_with_links(attack)
print(f"with link resolution: {'ALLOW' if ok else 'DENY '} — {why}")
print("\nText normalisation was not enough. The link had to be followed.")

## 4 · The control — all three layers, and a property test

A guard you have only tried on examples is a guard you have not tested. The version that earns trust is property-based: for *any* generated path, the resolved location must be inside the workspace and must not match a deny rule.

In [ ]:
import random
random.seed(3)

SEGMENTS = ["src", "..", ".", "vendor", "cache", "repo", "work", "root",
            ".ssh", ".aws", "id_rsa", "main.py", ".env", "etc", "shadow"]

def random_path():
    return "/" + "/".join(random.choice(SEGMENTS)
                          for _ in range(random.randint(1, 8)))

def final_location(path, links=SYMLINKS):
    return resolve_links(normalise(path), links)

violations, allowed_n = [], 0
for _ in range(20000):
    p = random_path()
    ok, _ = guard_with_links(p)
    if not ok:
        continue
    allowed_n += 1
    real = final_location(p)
    inside = real == "/work/repo" or real.startswith("/work/repo/")
    denied = any(fnmatch.fnmatch(real, g) for g in DENY_GLOBS)
    if not inside or denied:
        violations.append((p, real))

print(f"20000 random paths · {allowed_n} allowed · violations: {len(violations)}")
for p, real in violations[:5]:
    print("   ", p, "→", real)
assert not violations
print("\nProperty holds: every allowed path resolves inside the workspace")
print("and matches no deny rule — including through symlinks.")

In [ ]:
# And the same test against the buggy guard, to size the difference.
viol_buggy = 0
for _ in range(20000):
    p = random_path()
    if guard_buggy(p):
        real = final_location(p)
        if not (real == "/work/repo" or real.startswith("/work/repo/")):
            viol_buggy += 1
print(f"buggy prefix-check guard: {viol_buggy} paths allowed that escape the workspace")
assert viol_buggy > 0

## What you just proved

The correct guard gets all eight cases right; the buggy prefix check wrongly permits both traversals. The symlink attack passes textual normalisation and is only caught once links are resolved. The property test over 20,000 random paths reports zero violations for the three-layer guard and a non-zero count for the buggy one.

## Your turn

Find the path check in your own agent tooling. If it uses `startswith` or string concatenation without resolving links, run the property test above against it. It takes ten minutes and the result is not usually zero.

---

**Next → [A3.4 · Tool permission models](https://spbreed.github.io/cyber-commons/lessons/A3.4.html)**

[All lessons](https://spbreed.github.io/cyber-commons/lessons/) · [This lesson's page](https://spbreed.github.io/cyber-commons/lessons/A3.3.html) · [Source](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks/A3.3.ipynb)

*Cyber Commons — a free, open commons for Cyber AI.*